# 02 — Hadoop, HDFS e Localidade

**EP · Data Science Knowledge Base | Big Data e Processamento Distribuído | Notebook 02/16**

**Execução:** use o kernel Python 3, reinicie o kernel e execute as células em ordem. Laboratórios Spark são opcionais e desativados por padrão.

**Abordagem:** os exemplos locais verificam conceitos e não representam um cluster distribuído.


## Materiais complementares

- [Guia Técnico](./reference_materials/02_hdfs_hadoop.pdf)
- [Cheat Sheet](./cheat_sheets/cheat_sheet_hdfs_hadoop.pdf)

**Navegação:** os links são relativos à pasta deste notebook.

## Objetivos de aprendizagem
Descrever a finalidade do HDFS e avaliar limites do armazenamento distribuído.

Ao final, você deve conseguir interpretar os resultados, justificar as escolhas e explicar suas limitações.

## Fundamentação teórica
Blocos, replicação, NameNode/DataNode, disponibilidade e perda de nós.

**Atenção metodológica:** execução local facilita entendimento e depuração, mas não comprova ganhos de paralelismo em infraestrutura distribuída. Escalabilidade exige medir dados, nós, rede, memória, partições e operações de shuffle.

In [1]:
from pathlib import Path
import sys
DATASETS = Path('datasets')
assert DATASETS.is_dir(), 'Execute a partir da raiz da área 10 (pasta datasets não encontrada)'
print('Python:', sys.version.split()[0])
print('Pasta dos dados:', DATASETS.as_posix())

Python: 3.14.4
Pasta dos dados: datasets


## Demonstração 1 — construção e observação

In [2]:
from math import ceil
arquivo_mb, bloco_mb, replicacao = 525, 128, 3
blocos = ceil(arquivo_mb / bloco_mb)
ultimo_bloco_mb = arquivo_mb - (blocos-1)*bloco_mb
assert blocos == 5 and 0 < ultimo_bloco_mb <= bloco_mb
print({'blocos': blocos, 'réplicas_lógicas_de_blocos': blocos*replicacao,
       'último_bloco_mb': ultimo_bloco_mb,
       'armazenamento_aproximado_mb': arquivo_mb*replicacao})
print('Modelo aritmético: não representa política real de posicionamento ou overhead do HDFS.')


{'blocos': 5, 'réplicas_lógicas_de_blocos': 15, 'último_bloco_mb': 13, 'armazenamento_aproximado_mb': 1575}
Modelo aritmético: não representa política real de posicionamento ou overhead do HDFS.


## Interpretação
Observe o esquema, as unidades e os resultados. Antes de concluir que um algoritmo é eficiente, distinga **complexidade computacional**, **custo de comunicação** e **volume de dados**. Os cálculos locais ilustram a semântica; o Spark, quando disponível, ilustra a API distribuída.

## Demonstração 2 — validação e limitações

In [3]:
# Representação didática do posicionamento de blocos; não executa HDFS.
nos = ['n1','n2','n3','n4']
distribuicao = {f'bloco_{i}':[nos[(i+j)%len(nos)] for j in range(replicacao)] for i in range(blocos)}
assert all(len(set(rep))==replicacao for rep in distribuicao.values())
print(distribuicao)
print('A simulação garante nós distintos por bloco, mas não trata falha de rack ou políticas do HDFS.')


{'bloco_0': ['n1', 'n2', 'n3'], 'bloco_1': ['n2', 'n3', 'n4'], 'bloco_2': ['n3', 'n4', 'n1'], 'bloco_3': ['n4', 'n1', 'n2'], 'bloco_4': ['n1', 'n2', 'n3']}
A simulação garante nós distintos por bloco, mas não trata falha de rack ou políticas do HDFS.


## Boas práticas e limitações

- Execute com versões registradas, datasets identificados e caminhos relativos.
- Diferencie execução local, emulação lógica e infraestrutura distribuída real.
- Documente nulos, chaves, cardinalidade, custos e limitações das métricas.
- Não publique datasets ou resultados cuja redistribuição não esteja autorizada.
- Evite `collect()` de grandes tabelas e não confunda uma execução `local[*]` com cluster multinó.


## Exercícios
1. Explique que parte do exemplo poderia ser distribuída por chave ou partição e qual é o custo do shuffle.
2. Descreva uma validação de qualidade para os dados usados.
3. Indique um cenário em que o exemplo local deixaria de ser adequado.

## Respostas comentadas (orientativas)
1. Agregações por chave podem combinar resultados parciais, mas exigem redistribuir dados com a mesma chave; filtros e projeções usualmente evitam shuffle.
2. Verifique contagem, unicidade de chaves, percentual de valores ausentes, tipos, intervalos e consistência das unidades.
3. Quando dados, latência, concorrência ou exigências de falha excederem a capacidade de uma máquina; nesse caso, considere Spark/Hadoop e meça os custos adicionais.

## Síntese

O capítulo 02 reuniu conceitos e verificações de **replicação e falhas de nós**. Os exemplos de pequena escala não comprovam desempenho em cluster. Consulte apenas referências complementares efetivamente publicadas e documentadas.


## Revisão científica — auditoria e aplicação

**Ponto de atenção deste capítulo:** replicação e tolerância. Esta seção complementa os exemplos anteriores.

**Verificação conceitual.** Explique, com suas palavras, em qual etapa do fluxo de trabalho o tema **replicação e tolerância** pode produzir conclusões enganosas. Indique um indicador observável, um teste mínimo e uma limitação do procedimento.

**Exercício de extensão.** Reproduza o experimento central deste notebook alterando **um parâmetro por vez**. Compare com uma linha de base simples, reporte a variação da métrica pertinente e justifique o resultado. Quando houver treino/teste, qualquer etapa que aprenda parâmetros deve usar apenas o treino.

**Checklist de conclusão:** (1) entradas identificadas; (2) sementes registradas; (3) saídas interpretadas; (4) premissas explicitadas; (5) dados originais não sobrescritos.


### Exercício resolvido — protocolo mínimo de validação

**Enunciado.** Proponha uma comparação para **replicação e tolerância** sem contaminar a avaliação.

**Solução orientadora.** Defina uma hipótese falsificável, escolha um baseline simples, fixe uma divisão adequada ao problema, altere uma única decisão metodológica e compare métricas no conjunto reservado. Não transforme exemplos sintéticos em conclusões sobre populações reais. Documente a configuração e os casos de falha. Quando o tema não envolver predição, substitua o baseline por uma implementação de referência e compare correção, cobertura e custo.
